# Plan

### Cleaning data: Before running data through a model, it must be cleaned.
1. Drop columns that cannot be used in the model
2. Label encode the features to turn each unique weather type into a number

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# find a model to import (random forest)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [6]:
weather = pd.read_csv('weatherdataCMSE.csv')
weather = weather.rename(columns={'STATION': 'Station',
                                  'NAME': 'Name',
                                  'DATE': 'Date',
                                  'AWND': 'Avg Wind Speed',
                                  'PGTM': 'Peak Gust Time',
                                  'PRCP': 'Precipitation',
                                  'TAVG': 'Avg Temp',
                                  'TMAX': 'Max Temp',
                                  'TMIN': 'Min Temp',
                                  'WDF2': 'Direction of Fastest 2 Min Wind',
                                  'WDF5': 'Direction of Fastest 5 Sec Wind',
                                  'WSF2': 'Fastest 2 Min Wind Speed',
                                  'WSF5': 'Fastest 5 Sec Wind Speed',
                                  'WT01': 'Fog, Ice Fog, or Freezing Fog',
                                  'WT02': 'Heavy Fog, Heavy Freezing Fog',
                                  'WT03': 'Thunder',
                                  'WT08': 'Smoke or Haze'})
print("Percentage of Data Missing \n------------------------------------------------")
for column in weather.columns:
    missing = weather[column].isnull().sum()
    print(f"{column}: {(missing/len(weather))*100:.3f}%")

print("------------------------------------------------")

# Any columns with more than 5% of data missing can be dropped
columns_to_drop = []
for column in weather.columns:
    missing = weather[column].isnull().sum()
    missing_pct = (missing/len(weather))*100
    if missing_pct > 5:
        columns_to_drop.append(column)
print(f"Columns to drop: {columns_to_drop}")
print("------------------------------------------------")
display(weather.head())

Percentage of Data Missing 
------------------------------------------------
Station: 0.000%
Name: 0.000%
Date: 0.000%
Avg Wind Speed: 1.646%
Peak Gust Time: 1.621%
Precipitation: 0.633%
Avg Temp: 100.000%
Max Temp: 0.101%
Min Temp: 0.101%
Direction of Fastest 2 Min Wind: 1.570%
Direction of Fastest 5 Sec Wind: 1.570%
Fastest 2 Min Wind Speed: 1.570%
Fastest 5 Sec Wind Speed: 1.570%
Fog, Ice Fog, or Freezing Fog: 95.339%
Heavy Fog, Heavy Freezing Fog: 99.544%
Thunder: 99.037%
Smoke or Haze: 96.226%
------------------------------------------------
Columns to drop: ['Avg Temp', 'Fog, Ice Fog, or Freezing Fog', 'Heavy Fog, Heavy Freezing Fog', 'Thunder', 'Smoke or Haze']
------------------------------------------------


,Station,Name,Date,Avg Wind Speed,Peak Gust Time,Precipitation,Avg Temp,Max Temp,Min Temp,Direction of Fastest 2 Min Wind,Direction of Fastest 5 Sec Wind,Fastest 2 Min Wind Speed,Fastest 5 Sec Wind Speed,"Fog, Ice Fog, or Freezing Fog","Heavy Fog, Heavy Freezing Fog",Thunder,Smoke or Haze
0,USW00014822,"DETROIT CITY AIRPORT, MI US",2015-01-01,16.33,941.0,0.00,NaN,33.0,19.0,240.0,230.0,23.9,31.1,NaN,NaN,NaN,NaN
1,USW00014822,"DETROIT CITY AIRPORT, MI US",2015-01-02,6.71,1323.0,0.00,NaN,36.0,27.0,260.0,250.0,16.1,23.0,NaN,NaN,NaN,NaN
2,USW00014822,"DETROIT CITY AIRPORT, MI US",2015-01-03,6.71,1221.0,0.38,NaN,37.0,26.0,110.0,110.0,16.1,21.0,NaN,NaN,NaN,NaN
3,USW00014822,"DETROIT CITY AIRPORT, MI US",2015-01-04,9.84,2304.0,0.07,NaN,38.0,21.0,300.0,290.0,23.9,33.1,NaN,NaN,NaN,NaN
4,USW00014822,"DETROIT CITY AIRPORT, MI US",2015-01-05,13.20,445.0,0.00,NaN,21.0,8.0,310.0,320.0,23.0,36.9,NaN,NaN,NaN,NaN


## Drop bad columns and feature engineer new ones
> A bad column would be a variable that either does not give valuable information or has too many missing values to be able to contribute to the machine learning model
>
>
> Feature engineering is using raw data to create new data that can help describe a target variable

In [7]:
# Drop bad columns
weather = weather.drop(columns_to_drop, axis=1)
weather = weather.drop(['Name', 'Station'], axis=1)
weather = weather.dropna()

In [8]:
# Columns to add: 
# Did it rain the day before?
# Mean temperature (max temp - min temp / 2)
# Temperature range (max temp - min temp) aka "diurnal_range"

shifted_precip = weather['Precipitation'].shift(1)
precip_yesterday_bool = shifted_precip > 0
weather['Precipitation Yesterday'] = precip_yesterday_bool.astype(int)

weather['Mean Temp'] = (weather['Max Temp'] + weather['Min Temp']) / 2
weather['Diurnal Temp'] = weather['Max Temp'] - weather['Min Temp']

# Turn date into datetime object
weather['Date'] = pd.to_datetime(weather['Date'])

weather['Year'] = weather['Date'].dt.year
weather['Month'] = weather['Date'].dt.month
weather['Day'] = weather['Date'].dt.day
weather = weather.drop(columns=['Date'])

display(weather[['Precipitation', 'Precipitation Yesterday', 'Mean Temp', 'Diurnal Temp', 'Year', 'Month', 'Day']].head(10))

,Precipitation,Precipitation Yesterday,Mean Temp,Diurnal Temp,Year,Month,Day
0,0.00,0,26.0,14.0,2015,1,1
1,0.00,0,31.5,9.0,2015,1,2
2,0.38,0,31.5,11.0,2015,1,3
3,0.07,1,29.5,17.0,2015,1,4
4,0.00,1,14.5,13.0,2015,1,5
5,0.00,0,14.0,10.0,2015,1,6
6,0.01,0,12.0,12.0,2015,1,7
7,0.00,1,8.5,15.0,2015,1,8
8,0.00,0,12.5,13.0,2015,1,9
9,0.00,0,10.0,16.0,2015,1,10


In [9]:
features = weather.drop('Precipitation', axis=1).columns
X = weather[features]
y = weather['Precipitation']


train_X, test_X, train_y, test_y = train_test_split(
    X, y, 
    test_size=0.2,     
    random_state=42
)

print("TRAIN SHAPE:", train_X.shape, train_y.shape)
print("TEST SHAPE:", test_X.shape, test_y.shape)

TRAIN SHAPE: (3084, 14) (3084,)
TEST SHAPE: (772, 14) (772,)


In [10]:
forest_model = RandomForestRegressor(random_state=42)

forest_model.fit(train_X, train_y)

preds = forest_model.predict(test_X)

val_mae = mean_absolute_error(test_y, preds)

val_mae


0.10971295336787563